# 📊 MarketPulse 数据仪表盘

使用 MarketPulse API 快速查看 A 股市场四大核心指标。

In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

# 配置
API_BASE = "http://localhost:8898"
API_KEY = "mp-xxxxxxxx"
headers = {"X-API-Key": API_KEY}

def get(endpoint):
    return requests.get(f"{API_BASE}{endpoint}", headers=headers).json()["data"]

## 1️⃣ PE 分位 — 估值温度计

In [ ]:
pe = get("/api/v1/signal/pe-percentile")
print(f"当前PE: {pe['value']}")
print(f"历史分位: {pe['percentile']}%")
print(f"解读: {pe['interpretation']}")

fig, ax = plt.subplots(figsize=(8, 2))
ax.barh([0], [pe['percentile']], color='#ff6b6b' if pe['percentile']>70 else '#51cf66')
ax.set_xlim(0, 100)
ax.set_title(f"PE 分位: {pe['percentile']}% — {pe['interpretation']}")
ax.axvline(50, color='gray', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 2️⃣ 宏观综合评分

In [ ]:
macro = get("/api/v1/signal/macro-score")

# 提取子指标
subs = macro["sub_scores"]
names = {"m2": "M2增速", "pmi": "PMI", "cpi": "CPI", "shibor": "SHIBOR", "spread": "期限利差", "rmb": "汇率"}
items = [{"指标": names[k], "得分": round(v["score"]*100, 1)} for k, v in subs.items() if v["score"] is not None]
df = pd.DataFrame(items).set_index("指标")
df.plot(kind="barh", figsize=(8, 3), color="#339af0", legend=False)
plt.title(f"宏观评分: {macro['value']}/100 — {macro['interpretation']}")
plt.tight_layout()
plt.show()

## 3️⃣ 行业宽度 — Top/Bottom 5

In [ ]:
sector = get("/api/v1/signal/sector-breadth")
print(f"行业宽度: {sector['above_ma60_count']}/{sector['sector_count']} ({sector['value']*100:.1f}%)")
print(f"风险偏好: {sector['risk_appetite']}")
print()
print("🟢 Top 5 行业 (60日动量):")
for s in sector["top_5_sectors"]:
    print(f"  {s['name']}: {s['momentum_60d']:+.2f}%")
print()
print("🔴 Bottom 5 行业:")
for s in sector["bottom_5_sectors"]:
    print(f"  {s['name']}: {s['momentum_60d']:+.2f}%")

## 4️⃣ 综合快照

In [ ]:
erp = get("/api/v1/signal/erp")

snapshot = pd.DataFrame([
    {"指标": "PE分位", "数值": f"{pe['percentile']}%", "信号": pe['interpretation']},
    {"指标": "ERP", "数值": f"{erp['value']}%", "信号": erp['interpretation'][:8]},
    {"指标": "宏观评分", "数值": f"{macro['value']}/100", "信号": macro['interpretation']},
    {"指标": "行业宽度", "数值": f"{sector['value']*100:.1f}%", "信号": sector['interpretation']},
])
print(f"MarketPulse 市场快照 — {datetime.now().strftime('%Y-%m-%d')}")
print(snapshot.to_string(index=False))
print(f"\n⚠️ 仅供研究参考，不构成投资建议")